1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [2]:
load_dotenv(override=True)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [6]:
reader = PdfReader("AI_ML.pdf")
bussiness_info = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        bussiness_info += text

In [7]:
print(bussiness_info)

AI/ML Training Business — Business Information
About Our Business
We provide practical AI and Machine Learning training for students, beginners, and aspiring software and AI
professionals. Our focus is on learning concepts through coding practice and real-world projects.
Courses We Offer
Course
Level
Main Topics
Python Programming
Beginner
Python basics, functions, OOP, libraries
Data Science
Beginner–Intermediate
NumPy, Pandas, data analysis, visualization
Machine Learning
Intermediate
Regression, classification, clustering, model evaluation
Deep Learning
Intermediate
Neural networks, CNNs, model training
Generative AI
Intermediate
LLMs, prompting, APIs, AI applications
AI Agents
Intermediate
Tool calling, function calling, workflows, AI assistants
Machine Learning Program
 Duration: 12 weeks
 Level: Beginner to Intermediate
 Format: Online, instructor-led lessons with practical coding
 Projects: 5 practical projects
 Topics: Supervised learning, regression, classification, clust

In [9]:
system_prompt = f"""
You are an AI assistant for an AI/ML training business.

Use the following business information to answer user questions:

{bussiness_info}

Rules:
1. Answer only questions related to the business.
2. Use the provided business information.
3. Do not invent information that is not provided.
4. If a user wants to get in touch or provides their email,
   use the email recording tool.
5. Be helpful and concise.
"""

In [10]:
display(Markdown(system_prompt))



You are an AI assistant for an AI/ML training business.

Use the following business information to answer user questions:

AI/ML Training Business — Business Information
About Our Business
We provide practical AI and Machine Learning training for students, beginners, and aspiring software and AI
professionals. Our focus is on learning concepts through coding practice and real-world projects.
Courses We Offer
Course
Level
Main Topics
Python Programming
Beginner
Python basics, functions, OOP, libraries
Data Science
Beginner–Intermediate
NumPy, Pandas, data analysis, visualization
Machine Learning
Intermediate
Regression, classification, clustering, model evaluation
Deep Learning
Intermediate
Neural networks, CNNs, model training
Generative AI
Intermediate
LLMs, prompting, APIs, AI applications
AI Agents
Intermediate
Tool calling, function calling, workflows, AI assistants
Machine Learning Program
 Duration: 12 weeks
 Level: Beginner to Intermediate
 Format: Online, instructor-led lessons with practical coding
 Projects: 5 practical projects
 Topics: Supervised learning, regression, classification, clustering, feature engineering, model evaluation, and
basic deployment
Generative AI & AI Agents Program
 Duration: 8 weeks
 Level: Intermediate
 Topics: LLMs, prompting, APIs, tool/function calling, agents, and business automation
 Projects: 3 practical projects including a business AI assistant
Projects
Students work on practical projects such as a house price prediction system, customer-support AI assistant,
data analysis dashboard, and an AI agent that can use external tools.
Enrollment & Contact
Students can ask about courses, topics, projects, learning requirements, and enrollment. If a prospective
student wants to get in touch with the business, the assistant should ask for or accept their email address and
use the available email-recording tool.
Important Assistant Rule
The assistant should answer using the information in this document. It should not invent course details, prices,
guarantees, schedules, or other business information that is not provided. Questions unrelated to the business
should be politely declined.


Rules:
1. Answer only questions related to the business.
2. Use the provided business information.
3. Do not invent information that is not provided.
4. If a user wants to get in touch or provides their email,
   use the email recording tool.
5. Be helpful and concise.


In [11]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [12]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [13]:
tools = [{"type": "function", "function": record_email_tool_json}]


In [18]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages,tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
                response = gemini.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages,tools=tools)
            
      # LLM #1 ka final answer
    answer = response.choices[0].message.content

    # LLM #2 ke liye evaluation prompt
    evaluation_prompt = f"""
    You are a strict evaluator.

    Check whether the following answer is strictly related
    to the AI/ML training business.

    Answer only with:
    YES
    or
    NO

    Answer:
    {answer}
    """

    # LLM #2
    evaluation = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[
        {"role": "system", "content": "You are a strict evaluator."},
        {"role": "user", "content": evaluation_prompt}
    ]
)

    is_business_related = evaluation.choices[0].message.content.strip()

    if is_business_related == "YES":
        return answer
    else:
        return "I can only answer questions related to our AI/ML training business."

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


c:\Users\Abhishek\projects\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Abhishek\projects\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Abhishek\projects\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Abhishek\projects\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await